# GenoMed research: does Evo 2 agree with expert classifications?

Scores a **fixed, pre-chosen** set of 120 BRCA1 missense variants (60 expert-classified harmful, 60 harmless; ClinVar, *reviewed by expert panel*, GRCh37) with NVIDIA's hosted Evo 2, then compares the scores with the experts.

- The variant set and the analysis rules were fixed **before** scoring (`pipeline/research_variant_set.py`).
- Any failed call leaves the score **empty**; nothing is estimated.
- Research only: these scores never reach patients and never change a classification.

**Setup:** in Colab's 🔑 Secrets panel add `NVIDIA_API_KEY` (your `nvapi-…` key from build.nvidia.com) and turn on notebook access. No GPU needed.

In [ ]:
!git clone -q https://github.com/tasikokupatadze/genomed-ai
%cd genomed-ai
!pip install -q numpy requests

In [ ]:
import os
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
# The reference genome the variants are described on (GRCh37 = hg19), chromosome 17 only (~25 MB)
!wget -q -nc https://hgdownload.soe.ucsc.edu/goldenPath/hg19/chromosomes/chr17.fa.gz
!ls -lh chr17.fa.gz

## 1. One test call (probe)
Checks the endpoint and the response format before any real scoring. Default method **next_base** uses the documented hosted *generate* endpoint.
If it fails, copy the printed output to Claude.

In [ ]:
!python pipeline/score_variants.py --fasta chr17.fa.gz --probe --model evo2-40b

*Optional:* try the whole-window **forward** method (not in NVIDIA's hosted docs; the catalog lists `evo2-40b-forward`). Only use it for the run if this probe passes, and then use `--method forward` in **every** scoring command.

In [ ]:
# !python pipeline/score_variants.py --fasta chr17.fa.gz --probe --model evo2-40b --method forward

## 2. Score 5 variants first, then all of them
Safe to stop and re-run: finished variants are skipped.

In [ ]:
!python pipeline/score_variants.py --fasta chr17.fa.gz --model evo2-40b --limit 5

In [ ]:
!python pipeline/score_variants.py --fasta chr17.fa.gz --model evo2-40b

## 3. Compare with the experts

In [ ]:
!python pipeline/analyze_benchmark.py
import json
r = json.load(open("data/research/results.json"))
print(f"Model: {r['model']} · method: {r['method']} · scored {r['n_scored']} (failed {r['n_failed']})")
print(f"AUROC all: {r['auroc_all']} (95% CI {r['auroc_all_ci95']})")
print(f"AUROC test half: {r['auroc_test']} (95% CI {r['auroc_test_ci95']})")
print(f"Agreement on test half at cut-off {r['cutoff']}: {r['test_agreement']:.0%} of {r['n_test']}")

## 4. Put the results in the app
Download both files, then on GitHub open `data/research/` → **Add file → Upload files**, upload them and commit.
Vercel redeploys and the Research tab shows the real results (and stays on "Benchmark not yet run" until then).

In [ ]:
from google.colab import files
files.download("data/research/scores.csv")
files.download("data/research/results.json")